# Translate `mteb/StatcanDialogueDatasetRetrieval` to Spanish

This is a **retrieval** dataset (not flat sentences): `english-corpus` (documents),
`english-queries` (dialogue turns), `english-qrels` (relevance judgments). The corpus is
structured StatCan dataset metadata -- a `Title/Date range/Dimensions/Subject/Survey/
Frequency` header followed by category trees (`ID: 1, Parent: None, Name: Canada`).
Documents range up to **391,133 characters**, but `dev` and `test` corpora are
byte-identical (same 5907 docs), and across the whole corpus there are only **91,612
unique `Name` values** out of 592,259 occurrences (plus 5,899 titles, 552 surveys, 45
subjects, 13 frequencies, 2,853 dimension names) -- categories repeat constantly
("Canada", age groups, ...).

So instead of translating whole documents (infeasible at that size, and wasteful given
the repetition), this notebook:
1. Parses every corpus doc into an ordered list of segments (header fields, blank
   lines, dimension-block headers, `ID/Parent/Name` lines).
2. Collects the **global deduplicated set** of translatable strings across the whole
   corpus (titles, subjects, surveys, frequencies, dimension names, leaf names).
3. Translates that pool in batches (numbered-list prompts, ~40 strings/call) instead of
   591k+ individual calls.
4. Reconstructs every doc by substitution -- IDs, parent numbers, and date ranges are
   left untouched; the 6 header labels use a fixed EN->ES map. A handful of malformed
   lines (9 out of ~600k, from embedded newlines in the source data) don't fit the
   parser and are left untranslated -- negligible (<0.002%).

Parser/reconstruction fidelity was verified against the real corpus with an identity
translation (every string mapped to itself): 5,865/5,907 docs (99.3%) reconstruct
byte-for-byte identical to the source; the remaining 42 differ only in incidental
whitespace inside the source `Dimensions:` line (stray/non-breaking spaces already
present in the raw data) -- no content is altered.

`english-queries` (543 dev + 553 test rows, ~7,961 dialogue turns total -- informal,
typo-laden live chat between a citizen and a StatCan agent) is translated turn-by-turn,
like the earlier notebooks. `english-qrels` is just `query-id`/`corpus-id`/`score`
relations with no text, so it's copied through unchanged.

- `gemma` — `google/gemma-4-31B-it` on `http://localhost:8088/v1`
- `qwen`  — `Qwen/Qwen3.6-27B-FP8` on `http://localhost:8000/v1`

**Setup.** This repo's `uv` environment already has `datasets`; it does not have
`openai`. Launch this notebook with the extra dependency pulled in on the fly, without
touching `pyproject.toml`:

```bash
uv run --with openai --with ipykernel jupyter lab
```

Everything is checkpointed to `translations/statcan/*.jsonl`, so the notebook is safe
to interrupt and re-run — already-translated strings/turns are skipped.


In [1]:
import json
import random
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from openai import OpenAI
from tqdm.auto import tqdm


## Config

Only Spanish was requested for this dataset (unlike the earlier notebooks).

In [2]:
MODELS = {
    "gemma": {"base_url": "http://localhost:8088/v1", "model": "google/gemma-4-31B-it"},
    "qwen": {
        "base_url": "http://localhost:8000/v1",
        "model": "Qwen/Qwen3.6-27B-FP8",
        # Qwen3 is a hybrid-thinking model: without this it emits its chain-of-thought
        # as the actual response content instead of a final answer.
        "extra_body": {"chat_template_kwargs": {"enable_thinking": False}},
    },
}

LANGUAGES = {
    "es": "Spanish",
}

OUT_DIR = Path("translations/statcan")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 64
TEMPERATURE = 0.0

# Fixed header-label translations (Date range's VALUE is an ISO date range -- not
# translated -- but the label itself still needs localizing).
HEADER_LABELS = {
    "es": {
        "Title": "Título",
        "Date range": "Período",
        "Dimensions": "Dimensiones",
        "Subject": "Tema",
        "Survey": "Encuesta",
        "Frequency": "Frecuencia",
    },
}


In [3]:
clients = {name: OpenAI(base_url=cfg["base_url"], api_key="EMPTY") for name, cfg in MODELS.items()}

for name, client in clients.items():
    available = [m.id for m in client.models.list().data]
    print(f"{name} ({MODELS[name]['base_url']}): serving {available}")
    assert MODELS[name]["model"] in available, (
        f"{MODELS[name]['model']} not found on {name} server; available: {available}"
    )


gemma (http://localhost:8088/v1): serving ['google/gemma-4-31B-it']
qwen (http://localhost:8000/v1): serving ['Qwen/Qwen3.6-27B-FP8']


## Load the dataset

In [4]:
raw_corpus = load_dataset("mteb/StatcanDialogueDatasetRetrieval", "english-corpus")
raw_queries = load_dataset("mteb/StatcanDialogueDatasetRetrieval", "english-queries")
raw_qrels = load_dataset("mteb/StatcanDialogueDatasetRetrieval", "english-qrels")

# Confirmed dev/test corpora are identical -- verify and translate only once.
dev_map = {r["_id"]: r["text"] for r in raw_corpus["dev"]}
test_map = {r["_id"]: r["text"] for r in raw_corpus["test"]}
assert dev_map == test_map, "expected dev/test corpus to be identical"
print(f"corpus: {len(dev_map)} unique docs (shared by dev/test)")
print(f"queries: dev={len(raw_queries['dev'])}, test={len(raw_queries['test'])}")
print(f"qrels: dev={len(raw_qrels['dev'])}, test={len(raw_qrels['test'])}")


corpus: 5907 unique docs (shared by dev/test)
queries: dev=543, test=553
qrels: dev=799, test=870


## Parse each corpus doc into ordered segments

Line-based, order-preserving: replaying the segments reproduces the doc exactly except
for translated values, so blank-line placement / IDs / parent numbers are guaranteed
untouched.


In [5]:
HEADER_RE = re.compile(r"^(Title|Date range|Dimensions|Subject|Survey|Frequency):\s*(.*)$")
ID_LINE_RE = re.compile(r"^ID: (\S+), Parent: (\S+), Name: (.*)$")


def parse_doc(text):
    segments = []
    for line in text.split("\n"):
        if not line.strip():
            segments.append(("blank", line))
            continue
        m = ID_LINE_RE.match(line)
        if m:
            segments.append(("id", m.group(1), m.group(2), m.group(3)))
            continue
        m = HEADER_RE.match(line.strip())
        if m:
            segments.append(("header", m.group(1), m.group(2)))
            continue
        if line.rstrip().endswith(":"):
            segments.append(("block", line.rstrip()[:-1]))
            continue
        # Malformed line (embedded newline in the source Name/Title field) -- passed
        # through untranslated. Negligible: ~9 lines across the whole corpus.
        segments.append(("raw", line))
    return segments


parsed_docs = {doc_id: parse_doc(text) for doc_id, text in dev_map.items()}
raw_fallback_count = sum(1 for segs in parsed_docs.values() for s in segs if s[0] == "raw")
print(f"parsed {len(parsed_docs)} docs; {raw_fallback_count} untranslatable fallback lines")


parsed 5907 docs; 4 untranslatable fallback lines


## Collect the global deduplicated translation pool

Titles, subjects, surveys, frequencies, dimension names, and leaf `Name` values all
draw from ONE global lookup -- any string that appears in more than one role (e.g. a
dimension name that's also a leaf name elsewhere) only gets translated once.


In [6]:
all_strings = set()
for segs in parsed_docs.values():
    for seg in segs:
        if seg[0] == "header" and seg[1] in ("Title", "Subject", "Survey", "Frequency"):
            all_strings.add(seg[2])
        elif seg[0] == "block":
            all_strings.add(seg[1])
        elif seg[0] == "id":
            all_strings.add(seg[3])

all_strings.discard("")
all_strings = sorted(all_strings)
print(f"{len(all_strings)} unique strings to translate (deduplicated across the whole corpus)")


100683 unique strings to translate (deduplicated across the whole corpus)


## Per-item vocabulary translation

Each unique string is translated individually (one request per string) rather than
batched into numbered lists -- simpler, and avoids the batch-parsing/count-mismatch
retry logic entirely. Trade-off: ~91k+ individual requests instead of ~2.3k batched
ones -- more total requests, but each is small/fast, and at `MAX_WORKERS` concurrent
this is generally still quick on a local server.


In [7]:
EXAMPLES = {
    "es": [
        ("Geography", "Geografía"),
        ("Canada", "Canadá"),
        ("Age group", "Grupo de edad"),
        ("Labour Force Survey", "Encuesta de la Población Activa"),
    ],
}

VOCAB_SYSTEM_PROMPT = (
    "You are a professional translator localizing Statistics Canada open-data category "
    "names, dataset titles, and survey names for a data catalog. Translate the item from "
    "English into {lang_name}. Keep proper nouns as their correct {lang_name} exonym "
    "where one exists (e.g. 'Canada' -> its {lang_name} form; provinces/countries use "
    "their standard {lang_name} names). Keep numeric codes, acronyms in parentheses, and "
    "units of measure recognizable. Do not add, remove, or explain anything. "
    "Reply with ONLY the translation: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_vocab_examples_block(lang_code):
    lines = [f"EN: {en}\n{lang_code.upper()}: {es}" for en, es in EXAMPLES.get(lang_code, [])]
    return "\n\n".join(lines)


THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL | re.IGNORECASE)
REASONING_MARKERS = re.compile(
    r"^\s*(here'?s a thinking process|let'?s (think|analyze)|step \d|\d+\.\s+\*\*)",
    re.IGNORECASE,
)


def clean_translation(raw_out):
    out = THINK_RE.sub("", raw_out).strip()
    return out.strip('"').strip("'").strip()


def translate_vocab_item(client, model, text, lang_code, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = VOCAB_SYSTEM_PROMPT.format(lang_name=lang_name, examples_block=build_vocab_examples_block(lang_code))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature, extra_body=extra_body or {},
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Vocab translation failed for {text!r}: {last_err}")


### Checkpointed, concurrent per-item translation of the vocabulary pool

Same checkpoint file format as the batched version (`{"en": ..., "translated": ...}`
per line), so any progress already made by a previous run is still picked up as
cached -- switching from batched to per-item doesn't lose work already done.


In [8]:
def translate_vocab(strings, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    out_path = OUT_DIR / f"vocab_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[row["en"]] = row["translated"]

    todo = [s for s in strings if s not in done]
    print(f"[vocab/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_vocab_item, client, model, s, lang_code, extra_body): s
                for s in todo
            }
            bar = tqdm(as_completed(futures), total=len(futures), desc=f"{model_key}: vocab/{lang_code}", position=position, leave=True)
            for fut in bar:
                s = futures[fut]
                translated = fut.result()
                f.write(json.dumps({"en": s, "translated": translated}, ensure_ascii=False) + "\n")
                f.flush()
                done[s] = translated

    return done


## Smoke test

Translate a small sample of the vocabulary before committing to the full ~91k-string pool.

In [9]:
sample_strings = all_strings[:40]
for lang_code in LANGUAGES:
    for model_key in MODELS:
        result = translate_vocab(sample_strings, lang_code, model_key)
        for s in sample_strings[:8]:
            print(f"[{lang_code}/{model_key}] {s!r} -> {result[s]!r}")
        print()


[vocab/es/gemma] 8258 cached, 0 to translate via google/gemma-4-31B-it
[es/gemma] '     Perceived need for mental health care, needs partially met' -> 'Necesidad percibida de atención de salud mental, necesidades parcialmente satisfechas'
[es/gemma] '   Arab' -> 'Árabe'
[es/gemma] '   Arab/West Asian' -> 'Árabe/Asia Occidental'
[es/gemma] '   Black' -> 'Negro'
[es/gemma] '   Chinese' -> 'Chino'
[es/gemma] '   East or Southeast Asian' -> 'Asiático oriental o sudoriental'
[es/gemma] '   Filipino' -> 'Filipino'
[es/gemma] '   First Nations' -> 'Primeras Naciones'

[vocab/es/qwen] 100683 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8
[es/qwen] '     Perceived need for mental health care, needs partially met' -> 'Necesidad percibida de atención de salud mental, necesidades parcialmente satisfechas'
[es/qwen] '   Arab' -> 'Árabe'
[es/qwen] '   Arab/West Asian' -> 'Árabe/asiático occidental'
[es/qwen] '   Black' -> 'Negro'
[es/qwen] '   Chinese' -> 'Chino'
[es/qwen] '   East or Southeast Asi

## Full vocabulary translation

Same gemma/qwen-in-parallel pattern as the earlier notebooks. This is the expensive
step (~91k+ strings); everything else in this notebook is comparatively small.


In [10]:
def run_vocab_jobs(model_key, position):
    return {
        (lang_code, model_key): translate_vocab(all_strings, lang_code, model_key, position=position)
        for lang_code in LANGUAGES
    }


vocab = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {
        ex.submit(run_vocab_jobs, model_key, position): model_key
        for position, model_key in enumerate(MODELS)
    }
    for fut in as_completed(futures):
        vocab.update(fut.result())


[vocab/es/gemma] 8258 cached, 92425 to translate via google/gemma-4-31B-it
[vocab/es/qwen] 100683 cached, 0 to translate via Qwen/Qwen3.6-27B-FP8


gemma: vocab/es:   0%|          | 0/92425 [00:00<?, ?it/s]

## Reconstruct the translated corpus

Replays each doc's segments, substituting translated values via the vocabulary lookup.
IDs, parent numbers, date ranges, and the ~9 malformed fallback lines are left exactly
as in the source.


In [11]:
def format_header_line(label, value):
    # Matches the source exactly when value is empty: "Frequency:" not "Frequency: ".
    return f"{label}: {value}" if value else f"{label}:"


def reconstruct_doc(segments, lang_code, model_key):
    lookup = vocab[(lang_code, model_key)]
    labels = HEADER_LABELS[lang_code]
    lines = []
    for seg in segments:
        kind = seg[0]
        if kind == "blank":
            lines.append(seg[1])
        elif kind == "raw":
            lines.append(seg[1])
        elif kind == "id":
            _, id_, parent, name = seg
            lines.append(f"ID: {id_}, Parent: {parent}, Name: {lookup.get(name, name)}")
        elif kind == "block":
            dim_name = seg[1]
            lines.append(f"{lookup.get(dim_name, dim_name)}:")
        elif kind == "header":
            _, label, value = seg
            translated_label = labels.get(label, label)
            if label == "Date range":
                lines.append(format_header_line(translated_label, value))
            elif label == "Dimensions":
                dims = [d.strip() for d in value.split(",") if d.strip()]
                translated_value = ", ".join(lookup.get(d, d) for d in dims)
                lines.append(format_header_line(translated_label, translated_value))
            else:
                lines.append(format_header_line(translated_label, lookup.get(value, value)))
    return "\n".join(lines)


final_corpus = {}
for lang_code in LANGUAGES:
    for model_key in MODELS:
        translated_docs = {
            doc_id: reconstruct_doc(segs, lang_code, model_key) for doc_id, segs in parsed_docs.items()
        }
        final_corpus[(lang_code, model_key)] = translated_docs
        print(lang_code, model_key, "reconstructed", len(translated_docs), "docs")


es gemma reconstructed 5907 docs
es qwen reconstructed 5907 docs


## Spot-check corpus quality

In [12]:
lang_code = "es"
model_key = "gemma"
sample_ids = random.sample(list(dev_map), 3)
for doc_id in sample_ids:
    print("=" * 20, doc_id)
    print("EN:", dev_map[doc_id][:500])
    print(f"{lang_code.upper()} [{model_key}]:", final_corpus[(lang_code, model_key)][doc_id][:500])
    print()


==================== D21100107
EN: Title: Summary statistics for food services and drinking places (all establishments), by North American Industry Classification System (NAICS), inactive
Date range: 1998-01-01 to 2001-01-01
Dimensions: Geography, North American Industry Classification System (NAICS), Summary statistics
Subject: Business and consumer services and culture
Survey: Annual Survey of Service Industries: Food Services and Drinking Places
Frequency: Annual

Geography:
ID: 1, Parent: None, Name: Canada
ID: 2, Parent: 1, 
ES [gemma]: Título: Estadísticas resumidas de servicios de comida y bebidas (todos los establecimientos), por Sistema de Clasificación Industrial de América del Norte (SCIAN), inactivos
Período: 1998-01-01 to 2001-01-01
Dimensiones: Geografía, Sistema de Clasificación Industrial de América del Norte (SCIAN), Estadísticas resumidas
Tema: Servicios empresariales y al consumidor, y cultura
Encuesta: Encuesta Anual de Industrias de Servicios: Servicios de Alimento

## Translate queries (turn-by-turn)

Small volume (~8k turns total) and genuinely free-form dialogue, so this reuses the
simple per-item translation pattern from the earlier notebooks instead of batching.
Informal, typo-laden register is preserved rather than cleaned up.


In [13]:
QUERY_EXAMPLES = {
    "es": [
        (
            "hi, i was wondering if you vae any statistics on video game sales, ot high school drop out rates?",
            "hola, me preguntaba si tienen alguna estadistica sobre ventas de videojuegos, o tasas de abandono escolar?",
        ),
        (
            "Data for High School dropouts is compiled by the Provincial Education Ministry.",
            "Los datos sobre abandono escolar los recopila el Ministerio de Educacion provincial.",
        ),
    ],
}

QUERY_SYSTEM_PROMPT = (
    "You are a professional translator localizing a live chat between a citizen and a "
    "Statistics Canada data-request agent. Translate the message from English into "
    "{lang_name}. Keep the same meaning, tone, and register -- including typos, "
    "hesitations, or awkward phrasing -- but produce something that reads naturally to a "
    "native {lang_name} speaker. Do not add, remove, or explain anything. "
    "Reply with ONLY the translated sentence: no quotes, no notes, no alternatives.\n\n"
    "Examples:\n{examples_block}"
)


def build_query_examples_block(lang_code):
    lines = [f"EN: {en}\n{lang_code.upper()}: {es}" for en, es in QUERY_EXAMPLES.get(lang_code, [])]
    return "\n\n".join(lines)


# Reuses THINK_RE / REASONING_MARKERS / clean_translation defined earlier (vocabulary section).
def translate_turn(client, model, text, lang_code, extra_body=None, temperature=TEMPERATURE, max_retries=5):
    lang_name = LANGUAGES[lang_code]
    system = QUERY_SYSTEM_PROMPT.format(lang_name=lang_name, examples_block=build_query_examples_block(lang_code))
    messages = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages, temperature=temperature,
                extra_body=extra_body or {},
            )
            out = clean_translation(resp.choices[0].message.content)
            if out and not REASONING_MARKERS.search(out):
                return out
            last_err = RuntimeError(f"looks like leaked reasoning: {out[:120]!r}")
        except Exception as e:  # noqa: BLE001
            last_err = e
        time.sleep(min(2 ** attempt, 20))
    raise RuntimeError(f"Turn translation failed for {text!r}: {last_err}")


In [14]:
def translate_queries_split(split_name, dataset, lang_code, model_key, position=None):
    client = clients[model_key]
    model = MODELS[model_key]["model"]
    extra_body = MODELS[model_key].get("extra_body", {})

    units = [
        (row["_id"], turn_idx, turn["role"], turn["content"])
        for row in dataset
        for turn_idx, turn in enumerate(row["text"])
    ]

    out_path = OUT_DIR / f"queries_{split_name}_{lang_code}_{model_key}.jsonl"
    done = {}
    if out_path.exists():
        with out_path.open() as f:
            for line in f:
                row = json.loads(line)
                done[(row["query_id"], row["turn_idx"])] = row

    todo = [u for u in units if (u[0], u[1]) not in done]
    print(f"[queries/{split_name}/{lang_code}/{model_key}] {len(done)} cached, {len(todo)} to translate via {model}")

    if todo:
        with out_path.open("a") as f, ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {
                ex.submit(translate_turn, client, model, content, lang_code, extra_body): (query_id, turn_idx, role, content)
                for query_id, turn_idx, role, content in todo
            }
            bar = tqdm(as_completed(futures), total=len(futures), desc=f"{model_key}: queries/{split_name}", position=position, leave=True)
            for fut in bar:
                query_id, turn_idx, role, content = futures[fut]
                translated = fut.result()
                row = {"query_id": query_id, "turn_idx": turn_idx, "role": role, "content": content, "translated": translated}
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
                f.flush()
                done[(query_id, turn_idx)] = row

    return {(row["_id"], turn_idx): done[(row["_id"], turn_idx)]["translated"]
            for row in dataset for turn_idx in range(len(row["text"]))}


### Full queries run (gemma/qwen in parallel, same pattern as before)

In [15]:
def run_query_jobs(model_key, position):
    results = {}
    for split_name in ["dev", "test"]:
        for lang_code in LANGUAGES:
            results[(split_name, lang_code, model_key)] = translate_queries_split(
                split_name, raw_queries[split_name], lang_code, model_key, position=position
            )
    return results


translated_queries = {}
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    futures = {
        ex.submit(run_query_jobs, model_key, position): model_key
        for position, model_key in enumerate(MODELS)
    }
    for fut in as_completed(futures):
        translated_queries.update(fut.result())


[queries/dev/es/qwen] 0 cached, 3730 to translate via Qwen/Qwen3.6-27B-FP8[queries/dev/es/gemma] 0 cached, 3730 to translate via google/gemma-4-31B-it



gemma: queries/dev:   0%|          | 0/3730 [00:00<?, ?it/s]

qwen: queries/dev:   0%|          | 0/3730 [00:00<?, ?it/s]

[queries/test/es/gemma] 0 cached, 4231 to translate via google/gemma-4-31B-it


gemma: queries/test:   0%|          | 0/4231 [00:00<?, ?it/s]

[queries/test/es/qwen] 0 cached, 4231 to translate via Qwen/Qwen3.6-27B-FP8


qwen: queries/test:   0%|          | 0/4231 [00:00<?, ?it/s]

## Assemble final datasets and save

Mirrors the source repo's `{lang}-corpus` / `{lang}-queries` / `{lang}-qrels` config naming. `qrels` is copied through unchanged (no text, language-independent).

In [16]:
def build_queries_dataset(split_name, lang_code, model_key):
    lookup = translated_queries[(split_name, lang_code, model_key)]
    rows = []
    for row in raw_queries[split_name]:
        turns = [
            {"role": turn["role"], "content": lookup[(row["_id"], i)]}
            for i, turn in enumerate(row["text"])
        ]
        rows.append({"_id": row["_id"], "text": turns})
    return Dataset.from_list(rows)


final = {}
for lang_code in LANGUAGES:
    for model_key in MODELS:
        corpus_rows = [{"_id": doc_id, "text": text, "title": ""} for doc_id, text in final_corpus[(lang_code, model_key)].items()]
        corpus_ds = Dataset.from_list(corpus_rows)
        final[(lang_code, model_key, "corpus")] = DatasetDict({"dev": corpus_ds, "test": corpus_ds})
        final[(lang_code, model_key, "queries")] = DatasetDict({
            split_name: build_queries_dataset(split_name, lang_code, model_key) for split_name in ["dev", "test"]
        })
        final[(lang_code, model_key, "qrels")] = DatasetDict({
            split_name: raw_qrels[split_name] for split_name in ["dev", "test"]
        })
        print(lang_code, model_key, "corpus", len(corpus_rows), "docs")


es gemma corpus 5907 docs
es qwen corpus 5907 docs


In [17]:
SAVE_DIR = Path("translations/statcan_final")
for (lang_code, model_key, part), dd in final.items():
    dd.save_to_disk(str(SAVE_DIR / f"{lang_code}-{model_key}-{part}"))

# repo_id = "<your-username>/statcan-mt"
# for (lang_code, model_key, part), dd in final.items():
#     dd.push_to_hub(repo_id, config_name=f"{lang_code}-{model_key}-{part}")


Saving the dataset (0/1 shards):   0%|          | 0/5907 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5907 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/543 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/553 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/799 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/5907 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/543 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/553 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/799 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/870 [00:00<?, ? examples/s]